In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install pandas==1.1.5
!pip install wget==3.2
!pip install fasttext==0.9.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 41.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9683 sha256=cf914a30e27b3b92ef1740e8aaf9e1e257ddcd14482642d290c9be9842ab7eac
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [3]:
#necessary imports
import os
import pandas as pd
import wget
import tarfile

In [4]:
import os

# Buat folder DATAPATH
os.makedirs("DATAPATH", exist_ok=True)

# Hapus file download lama yang gagal
!rm -f DATAPATH/dbpedia_csv.tar.gz
!rm -f DATAPATH/dbpedia_csv.tar.gz.1

# Install gdown untuk download dari Google Drive
!pip -q install gdown

# Download dataset DBpedia
!gdown "https://drive.google.com/uc?id=0Bz8a_Dbh9QhbQ2Vic1kxMmZZQ1k" \
    -O DATAPATH/dbpedia_csv.tar.gz

# Cek ukuran file
print("\nUkuran file:")
!ls -lh DATAPATH/dbpedia_csv.tar.gz

# Extract dataset
!tar -xzf DATAPATH/dbpedia_csv.tar.gz -C DATAPATH

# Cek isi folder
print("\nIsi dataset:")
!find DATAPATH/dbpedia_csv -maxdepth 1 -type f -print

# Path dataset
data_path = "DATAPATH"

Downloading...
From (original): https://drive.google.com/uc?id=0Bz8a_Dbh9QhbQ2Vic1kxMmZZQ1k
From (redirected): https://drive.google.com/uc?id=0Bz8a_Dbh9QhbQ2Vic1kxMmZZQ1k&confirm=t&uuid=cb1719d9-d8af-44c9-89a5-601e38efffcf
To: /kaggle/working/DATAPATH/dbpedia_csv.tar.gz
100%|██████████████████████████████████████| 68.3M/68.3M [00:00<00:00, 73.0MB/s]

Ukuran file:
-rw-r--r-- 1 root root 66M Jun 20  2019 DATAPATH/dbpedia_csv.tar.gz

Isi dataset:
DATAPATH/dbpedia_csv/readme.txt
DATAPATH/dbpedia_csv/test.csv
DATAPATH/dbpedia_csv/classes.txt
DATAPATH/dbpedia_csv/train.csv


In [5]:
# Loading train data
train_file = data_path + '/dbpedia_csv/train.csv'
df = pd.read_csv(train_file, header=None, names=['class','name','description'])

# Loading test data
test_file = data_path + '/dbpedia_csv/test.csv'
df_test = pd.read_csv(test_file, header=None, names=['class','name','description'])

# Data we have
print("Train:{} Test:{}".format(df.shape, df_test.shape))

Train:(560000, 3) Test:(70000, 3)


In [6]:
# Since we have no clue about the classes lets build one
# Mapping from class number to class name
class_dict={
            1:'Company',
            2:'EducationalInstitution',
            3:'Artist',
            4:'Athlete',
            5:'OfficeHolder',
            6:'MeanOfTransportation',
            7:'Building',
            8:'NaturalPlace',
            9:'Village',
            10:'Animal',
            11:'Plant',
            12:'Album',
            13:'Film',
            14:'WrittenWork'
        }

# Mapping the classes
df['class_name'] = df['class'].map(class_dict)
df.head()

,class,name,description,class_name
0,1,E. D. Abbott Ltd,Abbott of Farnham E D Abbott Limited was a Br...,Company
1,1,Schwan-Stabilo,Schwan-STABILO is a German maker of pens for ...,Company
2,1,Q-workshop,Q-workshop is a Polish company located in Poz...,Company
3,1,Marvell Software Solutions Israel,Marvell Software Solutions Israel known as RA...,Company
4,1,Bergan Mercy Medical Center,Bergan Mercy Medical Center is a hospital loc...,Company


In [7]:
df["class_name"].value_counts()

class_name
Company                   40000
EducationalInstitution    40000
Artist                    40000
Athlete                   40000
OfficeHolder              40000
MeanOfTransportation      40000
Building                  40000
NaturalPlace              40000
Village                   40000
Animal                    40000
Plant                     40000
Album                     40000
Film                      40000
WrittenWork               40000
Name: count, dtype: int64

In [8]:
# Lets do some cleaning of this text
def clean_it(text,normalize=True):
    # Replacing possible issues with data. We can add or reduce the replacemtent in this chain
    s = str(text).replace(',',' ').replace('"','').replace('\'',' \' ').replace('.',' . ').replace('(',' ( ').\
            replace(')',' ) ').replace('!',' ! ').replace('?',' ? ').replace(':',' ').replace(';',' ').lower()
    
    # normalizing / encoding the text
    if normalize:
        s = s.normalize('NFKD').str.encode('ascii','ignore').str.decode('utf-8')
    
    return s

# Now lets define a small function where we can use above cleaning on datasets
def clean_df(data, cleanit= False, shuffleit=False, encodeit=False, label_prefix='__class__'):
    # Defining the new data
    df = data[['name','description']].copy(deep=True)
    df['class'] = label_prefix + data['class'].astype(str) + ' '
    
    # cleaning it
    if cleanit:
        df['name'] = df['name'].apply(lambda x: clean_it(x,encodeit))
        df['description'] = df['description'].apply(lambda x: clean_it(x,encodeit))
    
    # shuffling it
    if shuffleit:
        df.sample(frac=1).reset_index(drop=True)
            
    return df

In [9]:
%%time
# Transform the datasets using the above clean functions
df_train_cleaned = clean_df(df, True, True)
df_test_cleaned = clean_df(df_test, True, True)

CPU times: user 3.76 s, sys: 320 ms, total: 4.08 s
Wall time: 4.09 s


In [10]:
# Write files to disk as fastText classifier API reads files from disk.
train_file = data_path + '/dbpedia_train.csv'
df_train_cleaned.to_csv(train_file, header=None, index=False, columns=['class','name','description'] )

test_file = data_path + '/dbpedia_test.csv'
df_test_cleaned.to_csv(test_file, header=None, index=False, columns=['class','name','description'] )

In [11]:
%%time
## Using fastText for feature extraction and training
from fasttext import train_supervised 
"""fastText expects and training file (csv), a model name as input arguments.
label_prefix refers to the prefix before label string in the dataset.
default is __label__. In our dataset, it is __class__. 
There are several other parameters which can be seen in: 
https://pypi.org/project/fasttext/
"""
model = train_supervised(input=train_file, label="__class__", lr=1.0, epoch=75, loss='ova', wordNgrams=2, dim=200, thread=2, verbose=100)

Read 31M words
Number of words:  1116962
Number of labels: 14
Progress: 100.0% words/sec/thread:  717658 lr:  0.000111 avg.loss:  0.003208 ETA:   0h 0m 0s

CPU times: user 55min 19s, sys: 15.3 s, total: 55min 34s
Wall time: 27min 49s


Progress: 100.0% words/sec/thread:  717651 lr:  0.000000 avg.loss:  0.003208 ETA:   0h 0m 0s


In [12]:
for k in range(1,6):
    results = model.test(test_file,k=k)
    print(f"Test Samples: {results[0]} Precision@{k} : {results[1]*100:2.4f} Recall@{k} : {results[2]*100:2.4f}")

Test Samples: 70000 Precision@1 : 93.5829 Recall@1 : 93.5829
Test Samples: 70000 Precision@2 : 48.3821 Recall@2 : 96.7643
Test Samples: 70000 Precision@3 : 32.5124 Recall@3 : 97.5371
Test Samples: 70000 Precision@4 : 24.4204 Recall@4 : 97.6814
Test Samples: 70000 Precision@5 : 19.5546 Recall@5 : 97.7729
